<div dir="rtl">

# جلسه ۲۹: تحلیل اکتشافی داده (EDA)
## Exploratory Data Analysis

**بخش ۳: محاسبات علمی و کار با داده** — دوره‌ی پایتون برای علوم داده

مدرس: دکتر حسین حق‌بین

</div>

<div dir="rtl">

## 🎯 اهداف جلسه

در پایان این جلسه باید بتوانید:

- شاخص‌های **مرکز** (میانگین، میانه، مد) و **پراکندگی** (واریانس، انحراف معیار، چارک‌ها، IQR) را با Pandas حساب و تفسیر کنید و بدانید کدام در برابر داده‌ی پرت **مقاوم** است،
- چولگی و کشیدگی را بخوانید و تفاوت تعریف‌های Pandas و SciPy (مثل <code dir="ltr">ddof</code> و <code dir="ltr">bias</code>) را بدانید،
- با <code dir="ltr">scipy.stats</code> آمار توصیفی، <code dir="ltr">zscore</code>، <code dir="ltr">median&#95;abs&#95;deviation</code> و همبستگی‌های Pearson و Spearman را حساب کنید،
- مراحل یک **EDA** را به شکل یک چک‌لیست قابل تکرار اجرا کنید،
- روی یک داده‌ی واقعی (پنگوئن‌های Palmer) یک EDA کامل انجام دهید و یافته‌ها را به زبان ساده بنویسید.

> 📖 روش خواندن این جزوه و معنی علامت‌ها (💡 🔍 ❓ ⚠️ 📌 📝) در ابتدای جلسه‌ی ۱ توضیح داده شده است.

</div>

<div dir="rtl">

## فهرست مطالب

1. آمار توصیفی
2. آمار توصیفی با SciPy
3. مراحل EDA
4. EDA روی یک داده‌ی واقعی

</div>

<div dir="rtl">

## ۱. آمار توصیفی

### 💡 مفهوم

**تحلیل اکتشافی داده (EDA)** یعنی قبل از ساختن هر مدلی، داده را **بشناسید**: چه شکلی دارد، کجا مشکل دارد و چه الگوهایی در آن دیده می‌شود. اولین ابزار EDA، **آمار توصیفی** است: چند عدد که یک ستون را خلاصه می‌کنند.

| دسته | شاخص | متد Pandas | حساس به داده‌ی پرت؟ |
|---|---|---|---|
| مرکز | میانگین | <code dir="ltr">mean()</code> | بله |
| مرکز | میانه | <code dir="ltr">median()</code> | خیر |
| مرکز | مد (پرتکرارترین) | <code dir="ltr">mode()</code> | خیر |
| پراکندگی | واریانس / انحراف معیار | <code dir="ltr">var()</code> / <code dir="ltr">std()</code> | بله |
| پراکندگی | چارک‌ها و IQR | <code dir="ltr">quantile([.25, .75])</code> | خیر |
| پراکندگی | کمینه / بیشینه | <code dir="ltr">min()</code> / <code dir="ltr">max()</code> | بله |
| شکل | چولگی / کشیدگی | <code dir="ltr">skew()</code> / <code dir="ltr">kurt()</code> | بله |

یک داده‌ی ساختگی **چوله** (مثل درآمد: بیشتر افراد درآمد متوسط، چند نفر درآمد خیلی زیاد) می‌سازیم:

</div>

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(29)
income = pd.Series(np.round(rng.lognormal(mean=3.0, sigma=0.6, size=200), 1))

print("mean   :", round(income.mean(), 2))
print("median :", income.median())
print("std    :", round(income.std(), 2))
print("Q1, Q3 :", income.quantile([0.25, 0.75]).tolist())
print("skew   :", round(income.skew(), 2))
print("kurt   :", round(income.kurt(), 2))

<div dir="rtl">

### 🔍 کد را بخوان

- میانگین (<code dir="ltr">24.88</code>) از میانه (<code dir="ltr">20.85</code>) بزرگ‌تر است: چند مقدار خیلی بزرگ، میانگین را به راست می‌کشند. این نشانه‌ی **چولگی به راست** است و <code dir="ltr">skew</code> مثبت (<code dir="ltr">2.14</code>) همین را تأیید می‌کند.
- **چولگی (skewness)**: صفر یعنی متقارن، مثبت یعنی دم بلند در سمت راست، منفی یعنی دم بلند در سمت چپ.
- **کشیدگی (kurtosis)**: <code dir="ltr">kurt</code> در Pandas کشیدگی **اضافه** (excess) است؛ یعنی برای توزیع نرمال صفر است. مقدار مثبت (<code dir="ltr">6.58</code>) یعنی دم‌های سنگین‌تر از نرمال: مقادیر خیلی دور بیشتر از انتظار رخ می‌دهند.
- <code dir="ltr">quantile([0.25, 0.75])</code> چارک اول و سوم را می‌دهد؛ **IQR** فاصله‌ی این دو است (<code dir="ltr">30.35 - 13.0</code>) و پراکندگی نیمه‌ی میانی داده را نشان می‌دهد.

### ❓ پیش‌بینی کن

یک نفر با درآمد <code dir="ltr">2000</code> به داده اضافه می‌شود. قبل از اجرا حدس بزنید: میانگین، میانه و انحراف معیار هر کدام تقریباً چقدر تغییر می‌کنند؟

</div>

In [ ]:
income2 = pd.concat([income, pd.Series([2000.0])], ignore_index=True)
print("mean   :", round(income2.mean(), 2))
print("median :", income2.median())
print("std    :", round(income2.std(), 2))

<div dir="rtl">

🔍 یک مقدار از ۲۰۱ مقدار، میانگین را از <code dir="ltr">24.88</code> به <code dir="ltr">34.7</code> و انحراف معیار را از <code dir="ltr">17.71</code> به <code dir="ltr">140.43</code> رسانده، اما میانه تقریباً ثابت مانده است (<code dir="ltr">20.9</code>). به میانه، IQR و MAD (بخش ۲) آمار **مقاوم (robust)** می‌گویند: چند مقدار پرت یا اشتباه نمی‌تواند آن‌ها را خیلی جابه‌جا کند. وقتی داده چوله است یا پرت دارد، میانه را کنار میانگین گزارش کنید.

### ⚠️ اشتباه رایج: <code dir="ltr">ddof</code> در Pandas و NumPy

<code dir="ltr">Series.std()</code> در Pandas به‌طور پیش‌فرض بر <code dir="ltr">n - 1</code> تقسیم می‌کند (<code dir="ltr">ddof=1</code>، انحراف معیار **نمونه**)، اما <code dir="ltr">np.std</code> بر <code dir="ltr">n</code> (<code dir="ltr">ddof=0</code>). برای داده‌ی بزرگ تفاوت ناچیز است، برای داده‌ی کوچک نه:

</div>

In [ ]:
x = [2, 4, 4, 4, 5, 5, 7, 9]
print(np.std(x))
print(pd.Series(x).std())
print(np.std(x, ddof=1))

<div dir="rtl">

### 💡 ستون‌های رسته‌ای

برای ستون‌های رسته‌ای (متنی)، میانگین معنی ندارد؛ به جای آن **فراوانی** و **نسبت** هر رسته را می‌شماریم. <code dir="ltr">value&#95;counts(normalize=True)</code> نسبت‌ها را می‌دهد (مجموع ۱):

</div>

In [ ]:
colors = pd.Series(["red", "blue", "red", "green", "red", "blue"])
print(colors.value_counts())
print(colors.value_counts(normalize=True).round(3))
print(colors.mode().tolist())

<div dir="rtl">

🔍 <code dir="ltr">mode()</code> یک **Series** برمی‌گرداند، نه یک مقدار؛ چون ممکن است چند مقدار هم‌زمان پرتکرارترین باشند. برای گرفتن اولی: <code dir="ltr">colors.mode()[0]</code>.

</div>

<div dir="rtl">

## ۲. آمار توصیفی با SciPy

### 💡 مفهوم

کتابخانه‌ی **SciPy** (<code dir="ltr">scipy.stats</code>) ابزارهای آماری بیشتری دارد و روی آرایه‌های NumPy کار می‌کند. اگر نصب نیست، یک بار در Command Prompt اجرا کنید: <code dir="ltr">pip install scipy</code>.

| تابع | خروجی |
|---|---|
| <code dir="ltr">stats.describe(a)</code> | تعداد، کمینه و بیشینه، میانگین، واریانس، چولگی، کشیدگی؛ یک‌جا |
| <code dir="ltr">stats.skew(a)</code> / <code dir="ltr">stats.kurtosis(a)</code> | چولگی / کشیدگی |
| <code dir="ltr">stats.mode(a)</code> | مد و تعداد تکرار آن |
| <code dir="ltr">stats.iqr(a)</code> | دامنه‌ی میان‌چارکی |
| <code dir="ltr">stats.zscore(a)</code> | نمره‌ی استاندارد هر مقدار |
| <code dir="ltr">stats.median&#95;abs&#95;deviation(a)</code> | MAD: میانه‌ی قدر مطلق فاصله‌ها از میانه |
| <code dir="ltr">stats.pearsonr(x, y)</code> / <code dir="ltr">stats.spearmanr(x, y)</code> | ضریب همبستگی |

</div>

In [ ]:
from scipy import stats

data = np.array([12, 14, 15, 15, 16, 18, 20, 41])
print(stats.describe(data))
print("mode :", stats.mode(data))
print("iqr  :", stats.iqr(data))
print("MAD  :", stats.median_abs_deviation(data))

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">describe</code> یک شیء با چند فیلد برمی‌گرداند؛ هر فیلد را می‌توان جدا خواند: <code dir="ltr">stats.describe(data).mean</code>. واریانس آن (مثل Pandas) با <code dir="ltr">ddof=1</code> حساب شده است.
- **MAD** همتای مقاوم انحراف معیار است: به جای میانگینِ مجذور فاصله‌ها از میانگین، **میانه**‌ی قدر مطلق فاصله‌ها از **میانه** را می‌گیرد. مقدار <code dir="ltr">41</code> روی آن تقریباً اثری ندارد.

### ⚠️ اشتباه رایج: تعریف‌های متفاوت چولگی و کشیدگی

دو کتابخانه، یک نام، دو عدد متفاوت:

</div>

In [ ]:
s = pd.Series([2.0, 3, 3, 4, 4, 4, 5, 9, 15])
print(round(s.skew(), 4), round(stats.skew(s), 4), round(stats.skew(s, bias=False), 4))
print(round(s.kurt(), 4), round(stats.kurtosis(s), 4), round(stats.kurtosis(s, fisher=False), 4))

<div dir="rtl">

🔍 <code dir="ltr">stats.skew</code> به‌طور پیش‌فرض نسخه‌ی **اریب** (<code dir="ltr">bias=True</code>) را حساب می‌کند و Pandas نسخه‌ی اصلاح‌شده را؛ با <code dir="ltr">bias=False</code> دو عدد یکی می‌شوند (برای کشیدگی هم همین‌طور: <code dir="ltr">stats.kurtosis(s, bias=False)</code> همان <code dir="ltr">s.kurt()</code> است). برای کشیدگی، <code dir="ltr">fisher=True</code> (پیش‌فرض SciPy) کشیدگی اضافه است (نرمال = صفر) و <code dir="ltr">fisher=False</code> تعریف Pearson (نرمال = ۳). وقتی عددی گزارش می‌کنید، بگویید با کدام تعریف.

### 💡 نمره‌ی استاندارد (z-score)

<code dir="ltr">z = (x - mean) / std</code> می‌گوید هر مقدار چند انحراف معیار از میانگین فاصله دارد. <code dir="ltr">stats.zscore</code> به‌طور پیش‌فرض <code dir="ltr">ddof=0</code> به کار می‌برد (برخلاف <code dir="ltr">Series.std</code>):

</div>

In [ ]:
heights = np.array([160, 165, 170, 175, 180, 210])
z = stats.zscore(heights)
print(z.round(2))
print(heights[np.abs(z) > 2])

<div dir="rtl">

🔍 قاعده‌ی رایج «<code dir="ltr">&#124;z&#124; &gt; 3</code> یعنی پرت» فقط برای داده‌ی تقریباً نرمال و نه‌چندان کوچک معنی دارد. اینجا با شش مقدار، حتی <code dir="ltr">210</code> فقط به <code dir="ltr">z = 2.05</code> می‌رسد؛ با این تعداد داده، <code dir="ltr">&#124;z&#124;</code> اصلاً نمی‌تواند از <code dir="ltr">sqrt(5) ≈ 2.24</code> بیشتر شود (سؤال ۳ را ببینید). در جلسه‌ی ۲۶ روش IQR را برای پیدا کردن داده‌ی پرت دیدید؛ آن روش به میانگین و انحراف معیار وابسته نیست.

### 💡 همبستگی: Pearson و Spearman

- **Pearson** شدت رابطه‌ی **خطی** را می‌سنجد (بین <code dir="ltr">-1</code> و <code dir="ltr">1</code>).
- **Spearman** همان Pearson روی **رتبه‌ها**ست؛ رابطه‌ی **یکنوا** (همیشه صعودی یا همیشه نزولی) را می‌سنجد، حتی اگر خطی نباشد، و به داده‌ی پرت کمتر حساس است.

</div>

In [ ]:
x = np.linspace(0, 5, 50)
y = np.exp(x)

r, p = stats.pearsonr(x, y)
rho, p2 = stats.spearmanr(x, y)
print(round(r, 3), round(rho, 3))

<div dir="rtl">

### 🔍 کد را بخوان

<code dir="ltr">y</code> همیشه با <code dir="ltr">x</code> زیاد می‌شود (رابطه‌ی کاملاً یکنوا)، پس Spearman دقیقاً <code dir="ltr">1</code> است؛ اما رابطه خطی نیست و Pearson <code dir="ltr">0.854</code> است. هر دو تابع یک عدد دوم هم برمی‌گردانند (**p-value**)؛ معیاری که در درس‌های آمار می‌خوانید و اینجا از آن استفاده نمی‌کنیم.

### ❓ پیش‌بینی کن

اگر به جای <code dir="ltr">np.exp(x)</code> بنویسیم <code dir="ltr">y = (x - 2.5) &#42;&#42; 2</code> (یک سهمی)، هر دو ضریب تقریباً چقدر می‌شوند؟ آیا «همبستگی صفر» یعنی «هیچ رابطه‌ای نیست»؟

<details>
<summary><b>پاسخ</b></summary>

هر دو تقریباً صفر می‌شوند، چون نیمه‌ی چپ نزولی و نیمه‌ی راست صعودی است و اثرشان هم را خنثی می‌کند. اما <code dir="ltr">y</code> کاملاً با <code dir="ltr">x</code> تعیین می‌شود! همبستگی فقط رابطه‌ی خطی (Pearson) یا یکنوا (Spearman) را می‌بیند؛ برای همین در EDA همیشه **نمودار پراکندگی** را هم کنار عدد ببینید.

</details>

</div>

<div dir="rtl">

## ۳. مراحل EDA

### 💡 مفهوم

EDA یک کار پراکنده و سلیقه‌ای نیست؛ یک **فرایند** است که برای هر داده‌ی جدید تکرار می‌شود:

| مرحله | پرسش اصلی | ابزار |
|---|---|---|
| ۱. پرسش | می‌خواهیم با این داده به چه سؤالی جواب دهیم؟ | گفت‌وگو با صاحب داده |
| ۲. نمای کلی | چند سطر و ستون؟ هر ستون چه نوعی است؟ | <code dir="ltr">shape</code>، <code dir="ltr">dtypes</code>، <code dir="ltr">head</code> |
| ۳. کیفیت داده | مقدار گمشده، تکراری، نامعتبر؟ | <code dir="ltr">isna().sum()</code>، <code dir="ltr">duplicated()</code>، <code dir="ltr">describe</code> |
| ۴. تک‌متغیره | توزیع هر ستون چه شکلی است؟ | <code dir="ltr">describe</code>، <code dir="ltr">value&#95;counts</code>، هیستوگرام |
| ۵. دومتغیره | دو ستون با هم چه رابطه‌ای دارند؟ | <code dir="ltr">groupby</code>، <code dir="ltr">crosstab</code>، <code dir="ltr">corr</code>، boxplot، نمودار پراکندگی |
| ۶. چندمتغیره | آیا رابطه‌ی دو ستون به ستون سوم بستگی دارد؟ | رنگ‌بندی بر اساس گروه (<code dir="ltr">hue</code>)، heatmap، pairplot |
| ۷. یافته‌ها | چه چیزی یاد گرفتیم؟ چه سؤال تازه‌ای پیدا شد؟ | یادداشت متنی |
| ۸. قدم بعد | چه پاک‌سازی و چه مدلی لازم است؟ | برنامه‌ی کار |

مرحله‌های ۲ تا ۴ را می‌توان در یک تابع قابل استفاده‌ی مجدد گذاشت:

</div>

In [ ]:
def quick_eda(df, max_categories=10):
    print("shape:", df.shape)
    print("duplicated rows:", df.duplicated().sum())
    print()
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(1),
        "unique": df.nunique(),
    })
    print(summary)
    print()
    print(df.describe().round(2))
    for col in df.select_dtypes(exclude="number").columns:
        counts = df[col].value_counts(dropna=False)
        print()
        print(counts.head(max_categories))

In [ ]:
toy = pd.DataFrame({
    "age": [23, 35, 35, -1, 41, 29, 35],
    "city": ["Bushehr", "Tehran", "Tehran", "Shiraz", None, "Bushehr", "Tehran"],
    "score": [14.5, 17.0, 17.0, 12.0, 16.5, None, 17.0],
})
quick_eda(toy)

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">df.isna().mean()</code> کسر مقادیر گمشده‌ی هر ستون است (چون <code dir="ltr">True</code> یعنی ۱)؛ ضرب در ۱۰۰ درصد را می‌دهد.
- <code dir="ltr">duplicated()</code> سطرهایی را که **عیناً** تکرار سطر قبلی‌اند علامت می‌زند؛ اینجا سطرهای سوم و هفتم تکرار سطر دوم‌اند، پس <code dir="ltr">2</code> سطر تکراری داریم.
- <code dir="ltr">value&#95;counts(dropna=False)</code> مقدار گمشده را هم به‌عنوان یک رسته می‌شمارد تا از چشم نیفتد.
- در خروجی <code dir="ltr">describe</code>، کمینه‌ی <code dir="ltr">age</code> برابر <code dir="ltr">-1</code> است: سنی که ممکن نیست؛ احتمالاً کدی برای «نامعلوم». EDA دقیقاً برای دیدن همین چیزهاست.

### ⚠️ اشتباه رایج: پاک‌سازی قبل از دیدن

حذف سطرهای گمشده یا تکراری **قبل** از شمارش و فهمیدن آن‌ها، اطلاعات را از بین می‌برد: شاید گمشده بودن خودش الگو داشته باشد (مثلاً همه از یک شهر باشند). اول ببینید و یادداشت کنید، بعد تصمیم بگیرید (جلسه‌ی ۲۶).

</div>

<div dir="rtl">

## ۴. EDA روی یک داده‌ی واقعی

### 💡 داده و پرسش

داده‌ی **پنگوئن‌های Palmer** اندازه‌های بدن ۳۴۴ پنگوئن از سه گونه را در سه جزیره‌ی قطب جنوب (۲۰۰۷ تا ۲۰۰۹) دارد. این داده را دکتر Kristen Gorman و ایستگاه Palmer جمع‌آوری کرده‌اند و آزادانه منتشر شده است (CC0). فایل آن در پوشه‌ی <code dir="ltr">data</code> کنار نوت‌بوک‌هاست؛ اگر فقط نوت‌بوک را دانلود کرده‌اید، کد زیر آن را مستقیم از وبسایت درس می‌خواند.

**پرسش:** اندازه‌های بدن در گونه‌ها و جنس‌های مختلف چه تفاوتی دارند و آیا می‌توان گونه را از روی اندازه‌ها تشخیص داد؟

### مرحله‌ی ۲ و ۳: نمای کلی و کیفیت داده

</div>

In [ ]:
from pathlib import Path

LOCAL = Path("data/penguins.csv")
URL = "https://haghbinh.github.io/Python-Course/Notebooks/data/penguins.csv"
penguins = pd.read_csv(LOCAL if LOCAL.exists() else URL)

print(penguins.head())
quick_eda(penguins)

In [ ]:
print(penguins[penguins["bill_length_mm"].isna()])
print(penguins[penguins["sex"].isna()]["species"].value_counts())

<div dir="rtl">

### 🔍 کد را بخوان

- داده ۳۴۴ سطر و ۸ ستون دارد و سطر تکراری ندارد.
- چهار ستون اندازه هر کدام فقط ۲ مقدار گمشده دارند و خروجی دوم نشان می‌دهد این دو، **همان دو پنگوئن**اند که هیچ اندازه‌ای ندارند. ستون <code dir="ltr">sex</code> ۱۱ مقدار گمشده دارد (۶ پنگوئن Adelie و ۵ پنگوئن Gentoo).
- در <code dir="ltr">describe</code> مقدار نامعقولی دیده نمی‌شود: مثلاً جرم بدن بین <code dir="ltr">2700</code> و <code dir="ltr">6300</code> گرم است.

### مرحله‌ی ۴: تک‌متغیره

</div>

In [ ]:
print(penguins["species"].value_counts(normalize=True).round(3))

mass = penguins["body_mass_g"]
print("mean:", round(mass.mean(), 1), " median:", mass.median(), " mode:", mass.mode()[0])
print("skew:", round(mass.skew(), 3), " kurt:", round(mass.kurt(), 3))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=penguins, x="body_mass_g", bins=25, ax=ax)
ax.axvline(mass.mean(), color="red", label="mean")
ax.axvline(mass.median(), color="black", linestyle="--", label="median")
ax.set_xlabel("Body mass (g)")
ax.legend()
plt.show()

<p align="center"><img src="images/s29_mass_hist.png" width="620" alt="s29_mass_hist.png"></p>

<p align="center" dir="rtl"><i>توزیع جرم بدن همه‌ی پنگوئن‌ها: کمی چوله به راست؛ میانگین (قرمز) کمی بزرگ‌تر از میانه است.</i></p>

<div dir="rtl">

🔍 گونه‌ها متوازن نیستند: Adelie حدود ۴۴٪، Gentoo حدود ۳۶٪ و Chinstrap حدود ۲۰٪ داده است. جرم بدن کمی چوله به راست است (<code dir="ltr">skew = 0.47</code>؛ میانگین <code dir="ltr">4201.8</code> و میانه <code dir="ltr">4050</code>) و کشیدگی منفی (<code dir="ltr">-0.72</code>) دارد: توزیع از نرمال **پهن‌تر** است. این شکل اغلب نشانه‌ی **مخلوط چند گروه** است؛ در مرحله‌ی بعد می‌بینیم چرا.

### مرحله‌ی ۵: دومتغیره

</div>

In [ ]:
cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
print(penguins.groupby("species")[cols].mean().round(1))
print()
print(pd.crosstab(penguins["species"], penguins["island"]))
print()
print(penguins.groupby("sex")["body_mass_g"].mean().round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=penguins, x="species", y="body_mass_g", hue="sex", ax=ax)
ax.set_ylabel("Body mass (g)")
plt.show()

<p align="center"><img src="images/s29_mass_box.png" width="620" alt="s29_mass_box.png"></p>

<p align="center" dir="rtl"><i>جرم بدن بر اساس گونه و جنس: Gentoo به‌وضوح سنگین‌تر است و در هر گونه نرها سنگین‌ترند.</i></p>

<div dir="rtl">

### 🔍 کد را بخوان

- Gentoo به‌طور میانگین حدود ۱۳۰۰ گرم از دو گونه‌ی دیگر سنگین‌تر است و باله‌ی بلندتر و منقار کم‌عمق‌تری دارد؛ Chinstrap و Adelie در جرم شبیه‌اند ولی طول منقارشان فرق دارد (<code dir="ltr">48.8</code> در برابر <code dir="ltr">38.8</code> میلی‌متر). پس توزیع پهن جرم بدن در مرحله‌ی ۴، مخلوط یک گروه سنگین و دو گروه سبک‌تر بود.
- جدول <code dir="ltr">crosstab</code> یک الگوی مهم را نشان می‌دهد: Chinstrap **فقط** در جزیره‌ی Dream و Gentoo **فقط** در Biscoe دیده شده‌اند؛ فقط Adelie در هر سه جزیره هست.
- نرها به‌طور میانگین حدود ۷۰۰ گرم سنگین‌ترند و boxplot نشان می‌دهد این تفاوت در **هر سه** گونه برقرار است.

### مرحله‌ی ۶: چندمتغیره

</div>

In [ ]:
print(penguins[cols].corr().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(penguins[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
plt.show()

<p align="center"><img src="images/s29_corr_heatmap.png" width="480" alt="s29_corr_heatmap.png"></p>

<p align="center" dir="rtl"><i>ماتریس همبستگی Pearson چهار اندازه‌ی بدن.</i></p>

<div dir="rtl">

🔍 طول باله و جرم بدن همبستگی بسیار قوی دارند (<code dir="ltr">0.87</code>). اما یک عدد عجیب است: عمق منقار با جرم بدن همبستگی **منفی** دارد (<code dir="ltr">-0.47</code>)؛ یعنی پنگوئن‌های سنگین‌تر منقار کم‌عمق‌تری دارند؟

### ❓ پیش‌بینی کن

قبل از اجرای سلول بعد حدس بزنید: اگر همین همبستگی را **جداگانه در هر گونه** حساب کنیم، علامت آن چه می‌شود؟

</div>

In [ ]:
for species, group in penguins.dropna(subset=cols).groupby("species"):
    r = stats.pearsonr(group["bill_depth_mm"], group["body_mass_g"])[0]
    print(f"{species:<10} r = {r:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.scatterplot(data=penguins, x="bill_depth_mm", y="body_mass_g", hue="species", ax=ax)
ax.set_xlabel("Bill depth (mm)")
ax.set_ylabel("Body mass (g)")
plt.show()

<p align="center"><img src="images/s29_simpson.png" width="620" alt="s29_simpson.png"></p>

<p align="center" dir="rtl"><i>در هر گونه، منقار عمیق‌تر با جرم بیشتر همراه است؛ اما در کل داده، رابطه برعکس به نظر می‌رسد.</i></p>

<div dir="rtl">

### 🔍 کد را بخوان

در **هر سه** گونه همبستگی مثبت و نسبتاً قوی است (<code dir="ltr">0.58</code>، <code dir="ltr">0.60</code> و <code dir="ltr">0.72</code>)، اما در کل داده منفی است! دلیل: Gentoo هم سنگین‌ترین گونه است و هم کم‌عمق‌ترین منقار را دارد؛ وقتی گونه‌ها را با هم مخلوط می‌کنیم، تفاوت **بین** گروه‌ها بر رابطه‌ی **درون** هر گروه غلبه می‌کند. به این پدیده **پارادوکس سیمپسون** می‌گویند. درس مهم: یک همبستگی کلی را بدون نگاه به گروه‌ها تفسیر نکنید؛ این همان مرحله‌ی ۶ است.

### مرحله‌ی ۷ و ۸: یافته‌ها و قدم بعد

| # | یافته |
|---|---|
| ۱ | ۳۴۴ پنگوئن، ۸ ستون، بدون سطر تکراری؛ ۲ پنگوئن هیچ اندازه‌ای ندارند و جنس ۱۱ پنگوئن نامعلوم است. |
| ۲ | گونه‌ها نامتوازن‌اند (۴۴٪، ۳۶٪، ۲۰٪)؛ مدلی که همیشه Adelie بگوید ۴۴٪ دقت دارد و هر مدلی باید از این بهتر باشد. |
| ۳ | Gentoo با جرم بیشتر، باله‌ی بلندتر و منقار کم‌عمق‌تر به‌راحتی جدا می‌شود؛ Adelie و Chinstrap با طول منقار. |
| ۴ | نرها در هر گونه حدود ۷۰۰ گرم سنگین‌ترند؛ پس جنس هم باید در مقایسه‌ها در نظر گرفته شود. |
| ۵ | جزیره با گونه گره خورده است (Chinstrap فقط Dream، Gentoo فقط Biscoe)؛ اثر جزیره و اثر گونه را نمی‌توان از هم جدا کرد. |
| ۶ | همبستگی منفی کلی عمق منقار و جرم، اثر مخلوط شدن گونه‌هاست (پارادوکس سیمپسون). |

**قدم بعد:** حذف دو سطر بدون اندازه، تصمیم درباره‌ی جنس‌های گمشده (جلسه‌ی ۲۶)، و اگر هدف تشخیص گونه باشد، استفاده از طول منقار، عمق منقار و طول باله؛ با احتیاط درباره‌ی جزیره (سؤال ۷).

</div>

<div dir="rtl">

## 📌 آنچه باید بلد باشید

- شاخص‌های مرکز و پراکندگی؛ کدام‌ها مقاوم‌اند (میانه، IQR، MAD) و کدام‌ها به داده‌ی پرت حساس (میانگین، انحراف معیار، کمینه و بیشینه).
- رابطه‌ی میانگین و میانه با جهت چولگی؛ معنی کشیدگی اضافه.
- <code dir="ltr">ddof</code> در Pandas (<code dir="ltr">1</code>) و NumPy (<code dir="ltr">0</code>)؛ <code dir="ltr">bias</code> در <code dir="ltr">stats.skew</code> و <code dir="ltr">fisher</code> در <code dir="ltr">stats.kurtosis</code>.
- <code dir="ltr">value&#95;counts(normalize=True)</code> و <code dir="ltr">mode()</code> (که Series برمی‌گرداند) برای ستون‌های رسته‌ای.
- <code dir="ltr">stats.describe</code>، <code dir="ltr">iqr</code>، <code dir="ltr">zscore</code> (با <code dir="ltr">ddof=0</code>)، <code dir="ltr">median&#95;abs&#95;deviation</code>؛ محدودیت قاعده‌ی <code dir="ltr">&#124;z&#124; &gt; 3</code>.
- تفاوت Pearson (خطی) و Spearman (یکنوا، روی رتبه‌ها)؛ همبستگی صفر به معنی نبود رابطه نیست.
- هشت مرحله‌ی EDA و تابعی مثل <code dir="ltr">quick&#95;eda</code> برای مرحله‌های نمای کلی و کیفیت داده.
- خواندن خروجی <code dir="ltr">describe</code> برای پیدا کردن مقادیر نامعتبر؛ نامتوازن بودن کلاس‌ها و دقت خط پایه.
- پارادوکس سیمپسون و لزوم بررسی رابطه‌ها درون گروه‌ها.

</div>

<div dir="rtl">

## 📝 سؤالات مرور / نمونه سؤال امتحان

> اول خودتان جواب دهید (روی کاغذ، بدون اجرای کد)، بعد پاسخ را باز کنید و در آخر کد را اجرا کنید.

### سؤال ۱ — آمار مقاوم

حقوق ماهانه‌ی هفت کارمند (میلیون تومان) این است: <code dir="ltr">[12, 14, 15, 15, 16, 18, 20]</code>. حقوق نفر آخر به‌اشتباه به جای <code dir="ltr">20</code> عدد <code dir="ltr">200</code> ثبت شده است.

الف) برای هر دو حالت (درست و اشتباه) میانگین، میانه، IQR (با <code dir="ltr">stats.iqr</code>) و MAD (با <code dir="ltr">stats.median&#95;abs&#95;deviation</code>) را حساب کنید.

ب) انحراف معیار (<code dir="ltr">ddof=1</code>) در دو حالت <code dir="ltr">2.63</code> و <code dir="ltr">69.95</code> است. کدام شاخص‌ها تغییر کردند و کدام نه؟ برای گزارش «حقوق معمول» در این شرکت چه شاخصی پیشنهاد می‌کنید؟

ج) یک **مدل رگرسیون** که با تابع زیان MSE آموزش می‌بیند، به کدام یک از این دو شاخص (میانگین یا میانه) «نزدیک‌تر» است؟ چرا داده‌ی پرت برای آن خطرناک است؟

<details>
<summary><b>پاسخ</b></summary>

الف)

| | میانگین | میانه | IQR | MAD |
|---|---|---|---|---|
| درست | <code dir="ltr">15.71</code> | <code dir="ltr">15</code> | <code dir="ltr">2.5</code> | <code dir="ltr">1</code> |
| اشتباه | <code dir="ltr">41.43</code> | <code dir="ltr">15</code> | <code dir="ltr">2.5</code> | <code dir="ltr">1</code> |

ب) میانگین (از <code dir="ltr">15.7</code> به <code dir="ltr">41.4</code>) و انحراف معیار (۲۶ برابر شد) به‌شدت تغییر کردند؛ میانه، IQR و MAD هیچ تغییری نکردند، چون فقط به رتبه‌ی وسط و چارک‌ها وابسته‌اند و <code dir="ltr">200</code> همچنان «بزرگ‌ترین مقدار» است. برای حقوق معمول، **میانه** (و برای پراکندگی، IQR) را گزارش کنید؛ میانگین ۴۱ میلیون تصویر کاملاً غلطی می‌دهد که حتی یک نفر هم نزدیک آن نیست.

ج) مقدار ثابتی که MSE را کمینه می‌کند **میانگین** است (و مقداری که MAE را کمینه می‌کند، میانه). چون خطا **مجذور** می‌شود، یک مقدار پرت سهم بسیار بزرگی در زیان دارد و ضرایب مدل را به سمت خودش می‌کشد؛ برای همین قبل از مدل‌سازی باید پرت‌ها را در EDA دید (یا از زیان مقاوم‌تری مثل MAE یا Huber استفاده کرد).

</details>

### سؤال ۲ — <code dir="ltr">ddof</code>

الف) خروجی کد زیر چیست؟

</div>

```python
x = [3, 5]
print(np.std(x), pd.Series(x).std())
print(stats.zscore(x))
```

<div dir="rtl">

ب) کدام برای «انحراف معیار جامعه» و کدام برای «برآورد انحراف معیار جامعه از روی نمونه» مناسب است؟ چرا تقسیم بر <code dir="ltr">n - 1</code> برآورد بهتری است؟

ج) دانشجویی ستون‌های داده را با <code dir="ltr">(df - df.mean()) / df.std()</code> استاندارد کرده و همکارش با <code dir="ltr">stats.zscore(df)</code>. آیا نتیجه‌ها یکی است؟ چه وقت تفاوت مهم می‌شود؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">1.0 1.4142135623730951</code> و <code dir="ltr">[-1.  1.]</code>. (<code dir="ltr">zscore</code> با <code dir="ltr">ddof=0</code> کار می‌کند، پس دو مقدار دقیقاً یک انحراف معیار از میانگین فاصله دارند.)

ب) <code dir="ltr">ddof=0</code> (تقسیم بر <code dir="ltr">n</code>) انحراف معیار خود همین داده‌هاست، اگر همه‌ی جامعه را داشته باشیم. <code dir="ltr">ddof=1</code> برآورد انحراف معیار جامعه از روی نمونه است. فاصله‌ها از **میانگین نمونه** اندازه‌گیری می‌شوند که خودش به داده‌ها نزدیک‌ترین نقطه است؛ پس مجموع مجذور فاصله‌ها کمی کوچک‌تر از مقدار واقعی است و تقسیم بر <code dir="ltr">n - 1</code> این کم‌برآوردی را جبران می‌کند.

ج) نه دقیقاً: <code dir="ltr">df.std()</code> در Pandas <code dir="ltr">ddof=1</code> دارد و <code dir="ltr">zscore</code> <code dir="ltr">ddof=0</code>؛ نتیجه‌ها با ضریب ثابت <code dir="ltr">sqrt((n - 1) / n)</code> فرق دارند. برای چند هزار سطر این ضریب تقریباً ۱ است و مهم نیست؛ برای داده‌ی کوچک (مثلاً ۵ نمونه، ضریب <code dir="ltr">0.89</code>) فرق محسوس است. مهم‌تر این است که میانگین و انحراف معیار فقط روی داده‌ی **آموزش** حساب شوند (جلسه‌ی ۲۶).

</details>

### سؤال ۳ — z-score و پوشاندن پرت

در یک سنسور، این هشت اندازه ثبت شده است: <code dir="ltr">[10, 11, 9, 10, 12, 10, 11, 60]</code>.

الف) <code dir="ltr">stats.zscore</code> را برای آن حساب کنید. آیا قاعده‌ی <code dir="ltr">&#124;z&#124; &gt; 3</code> مقدار <code dir="ltr">60</code> را پرت تشخیص می‌دهد؟

ب) نشان دهید که در <code dir="ltr">n</code> داده، بزرگ‌ترین <code dir="ltr">&#124;z&#124;</code> ممکن (با <code dir="ltr">ddof=0</code>) برابر <code dir="ltr">sqrt(n - 1)</code> است. (راهنمایی: حالتی را در نظر بگیرید که <code dir="ltr">n - 1</code> مقدار صفر و یک مقدار <code dir="ltr">a</code> باشد.) برای <code dir="ltr">n = 8</code> این عدد چقدر است و چه نتیجه‌ای می‌گیرید؟

ج) **z-score مقاوم** را با میانه و MAD حساب کنید: <code dir="ltr">(x - median) / MAD</code> که در آن <code dir="ltr">MAD = stats.median&#95;abs&#95;deviation(x, scale="normal")</code>. حالا <code dir="ltr">60</code> چه نمره‌ای می‌گیرد؟

<details>
<summary><b>پاسخ</b></summary>

الف)

<div dir="ltr">

```text
[-0.4  -0.34 -0.46 -0.4  -0.28 -0.4  -0.34  2.64]
```

</div>

نه؛ <code dir="ltr">2.64 &lt; 3</code>. خود مقدار پرت، میانگین را بالا و انحراف معیار را بزرگ کرده و با این کار خودش را «پنهان» کرده است (**masking**).

ب) بیشترین <code dir="ltr">&#124;z&#124;</code> وقتی رخ می‌دهد که <code dir="ltr">n - 1</code> مقدار برابر باشند و فقط یکی دور باشد. با <code dir="ltr">n - 1</code> صفر و یک مقدار <code dir="ltr">a</code>: میانگین <code dir="ltr">a / n</code> است و فاصله‌ی <code dir="ltr">a</code> از میانگین <code dir="ltr">a (n - 1) / n</code>. واریانس (با <code dir="ltr">ddof=0</code>) برابر <code dir="ltr">a² (n - 1) / n²</code> و انحراف معیار <code dir="ltr">a sqrt(n - 1) / n</code> است؛ پس <code dir="ltr">z = sqrt(n - 1)</code>. برای <code dir="ltr">n = 8</code> این حد <code dir="ltr">2.65</code> است و مقدار ما (<code dir="ltr">2.64</code>) تقریباً به همین سقف رسیده، چون هفت مقدار دیگر به هم نزدیک‌اند. (با <code dir="ltr">ddof=1</code> سقف <code dir="ltr">(n - 1) / sqrt(n)</code> یعنی <code dir="ltr">2.47</code> است.) نتیجه: با ۸ داده، رسیدن به <code dir="ltr">&#124;z&#124; &gt; 3</code> **غیرممکن** است؛ قاعده‌ی <code dir="ltr">&#124;z&#124; &gt; 3</code> برای داده‌ی کوچک بی‌معنی است.

ج) میانه <code dir="ltr">10.5</code> و MAD نرمال‌شده <code dir="ltr">0.74</code> است و <code dir="ltr">60</code> نمره‌ی حدود <code dir="ltr">66.8</code> می‌گیرد؛ به‌وضوح پرت. میانه و MAD تحت تأثیر خود مقدار پرت قرار نمی‌گیرند.

</details>

### سؤال ۴ — Pearson یا Spearman؟

الف) برای <code dir="ltr">x = np.arange(1, 11)</code> و <code dir="ltr">y = x &#42;&#42; 3</code>، دو ضریب Pearson و Spearman را (تا سه رقم اعشار) پیش‌بینی و بعد حساب کنید.

ب) حالا یک نقطه‌ی پرت <code dir="ltr">(11, -500)</code> به داده اضافه کنید. هر دو ضریب چقدر می‌شوند؟ کدام بیشتر آسیب دیده و چرا؟

ج) در انتخاب ویژگی (feature selection) برای یک مدل، دانشجویی فقط ویژگی‌هایی را نگه داشته که <code dir="ltr">&#124;Pearson&#124; &gt; 0.3</code> با هدف دارند. دو حالت مثال بزنید که این روش ویژگی مفیدی را دور می‌اندازد.

<details>
<summary><b>پاسخ</b></summary>

الف) Pearson <code dir="ltr">0.928</code> و Spearman <code dir="ltr">1.0</code>: رابطه کاملاً صعودی است (رتبه‌ها یکسان‌اند) ولی خطی نیست.

ب) Pearson <code dir="ltr">0.348</code> و Spearman <code dir="ltr">0.5</code>. Pearson بسیار بیشتر افت کرده، چون با **مقدار** نقاط کار می‌کند و <code dir="ltr">-500</code> فاصله‌ی عظیمی از خط دارد؛ Spearman فقط رتبه را می‌بیند و برای آن، <code dir="ltr">-500</code> صرفاً «کوچک‌ترین رتبه» است.

ج) (۱) رابطه‌ی غیریکنوا مثل سهمی یا U-شکل (مثلاً اثر دما بر مصرف برق که در هوای خیلی سرد و خیلی گرم زیاد است): Pearson نزدیک صفر ولی ویژگی بسیار مفید. (۲) ویژگی‌ای که فقط **همراه** ویژگی دیگر اثر دارد (اثر متقابل) یا فقط درون گروه‌ها اثر دارد؛ مثل عمق منقار در بخش ۴ که همبستگی کلی‌اش منفی و درون هر گونه مثبت است. همبستگی دوبه‌دو فقط یک فیلتر اولیه است.

</details>

### سؤال ۵ — خواندن <code dir="ltr">describe</code>

خروجی <code dir="ltr">describe()</code> یک داده‌ی مشتریان این است:

</div>

```text
stat    age      income   n_orders   country_code
count   980.0    1000.0    1000.0         1000.0
mean     41.2    5310.4      3.1           1.0
std      15.8   12890.7      2.4           0.0
min      -1.0       0.0      0.0           1.0
25%      28.0    1800.0      1.0           1.0
50%      39.0    3100.0      3.0           1.0
75%      52.0    5200.0      4.0           1.0
max     999.0  410000.0     31.0           1.0
```

<div dir="rtl">

الف) دست‌کم پنج مشکل یا نکته‌ی مشکوک در این جدول پیدا کنید و برای هر کدام بگویید قدم بعدی چیست.

ب) تابع <code dir="ltr">quick&#95;eda</code> بخش ۳ را طوری گسترش دهید که ستون‌های **ثابت** (فقط یک مقدار متمایز) و ستون‌هایی را که بیش از <code dir="ltr">50%</code> مقدار گمشده دارند، جداگانه چاپ کند.

<details>
<summary><b>پاسخ</b></summary>

الف)

- <code dir="ltr">age</code>: <code dir="ltr">count = 980</code> یعنی ۲۰ مقدار گمشده؛ <code dir="ltr">min = -1</code> و <code dir="ltr">max = 999</code> تقریباً قطعاً **کد** برای «نامعلوم» هستند و باید به <code dir="ltr">NaN</code> تبدیل شوند، وگرنه میانگین و انحراف معیار را خراب می‌کنند.
- <code dir="ltr">income</code>: میانگین (<code dir="ltr">5310</code>) بسیار بیشتر از میانه (<code dir="ltr">3100</code>) و انحراف معیار دو برابر میانگین: توزیع به‌شدت چوله به راست، با بیشینه‌ی <code dir="ltr">410000</code>. باید بررسی شود (پرت واقعی یا خطای ورود؟) و برای خلاصه‌سازی میانه به کار رود؛ شاید تبدیل لگاریتمی لازم باشد.
- <code dir="ltr">income</code> با <code dir="ltr">min = 0</code>: درآمد صفر واقعی است یا مقدار گمشده؟ تعدادش را بشمارید.
- <code dir="ltr">n&#95;orders</code> با <code dir="ltr">max = 31</code> در حالی که چارک سوم <code dir="ltr">4</code> است: یک یا چند مشتری بسیار فعال (یا یک حساب تجاری/آزمایشی).
- <code dir="ltr">country&#95;code</code>: <code dir="ltr">std = 0</code> و <code dir="ltr">min = max = 1</code>؛ ستون **ثابت** است و هیچ اطلاعاتی ندارد؛ یا همه از یک کشورند یا خروجی اشتباه گرفته شده. برای مدل بی‌فایده است.

ب)

<div dir="ltr">

```python
def quick_eda(df, max_categories=10):
    # ... the original body ...
    constant = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
    mostly_missing = [c for c in df.columns if df[c].isna().mean() > 0.5]
    print("constant columns:", constant)
    print("columns with > 50% missing:", mostly_missing)
```

</div>

(<code dir="ltr">nunique(dropna=False)</code> تا ستونی که فقط یک مقدار و چند <code dir="ltr">NaN</code> دارد، ثابت حساب نشود.)

</details>

### سؤال ۶ — کلاس‌های نامتوازن

الف) در داده‌ای از تراکنش‌های بانکی، <code dir="ltr">99%</code> تراکنش‌ها سالم و <code dir="ltr">1%</code> تقلب‌اند. مدلی که همیشه «سالم» می‌گوید، چه دقتی (accuracy) دارد؟ چرا این عدد گمراه‌کننده است؟

ب) در داده‌ی پنگوئن‌ها، دقت مدل «همیشه پرتکرارترین گونه» را با یک خط کد Pandas حساب کنید.

ج) **دقت متوازن (balanced accuracy)** میانگین recall تک‌تک کلاس‌هاست. دقت متوازن مدل «همیشه پرتکرارترین کلاس» برای داده‌ای با <code dir="ltr">k</code> کلاس چقدر است؟ چرا برای داده‌ی نامتوازن معیار بهتری است؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">99%</code>؛ اما مدل حتی یک تقلب را هم پیدا نکرده است، یعنی برای هدف اصلی کاملاً بی‌فایده است. در داده‌ی نامتوازن، accuracy عمدتاً عملکرد روی کلاس بزرگ را نشان می‌دهد. EDA باید نسبت کلاس‌ها (<code dir="ltr">value&#95;counts(normalize=True)</code>) را **قبل** از انتخاب معیار ارزیابی نشان دهد.

ب) <code dir="ltr">penguins["species"].value&#95;counts(normalize=True).max()</code> که حدود <code dir="ltr">0.442</code> است.

ج) recall کلاس پرتکرار <code dir="ltr">1</code> و recall بقیه‌ی کلاس‌ها <code dir="ltr">0</code> است؛ پس دقت متوازن <code dir="ltr">1 / k</code> است (برای پنگوئن‌ها <code dir="ltr">0.333</code> و برای تراکنش‌ها <code dir="ltr">0.5</code>)، مستقل از اینکه کلاس‌ها چقدر نامتوازن باشند. چون هر کلاس سهم برابر دارد، مدل نمی‌تواند با نادیده گرفتن کلاس کوچک امتیاز خوب بگیرد.

</details>

### سؤال ۷ — EDA چه چیزی را نمی‌تواند ثابت کند؟

الف) در بخش ۴ دیدیم Chinstrap فقط در جزیره‌ی Dream و Gentoo فقط در Biscoe دیده شده است. اگر ستون <code dir="ltr">island</code> را به یک مدل تشخیص گونه بدهیم، دقت روی همین داده بالا می‌رود. این کار مشکلی دارد؟ اگر مدل قرار باشد روی پنگوئن‌های یک جزیره‌ی **جدید** استفاده شود چطور؟

ب) کدام یک از جمله‌های زیر را EDA روی این داده **می‌تواند** پشتیبانی کند و کدام را **نمی‌تواند**؟ چرا؟

1. «در این نمونه، نرها در هر سه گونه به‌طور میانگین سنگین‌تر از ماده‌ها هستند.»
2. «زندگی در جزیره‌ی Biscoe باعث سنگین‌تر شدن پنگوئن‌ها می‌شود.»
3. «عمق منقار بیشتر باعث کاهش جرم بدن می‌شود.»

ج) در یک داده‌ی پزشکی، ستون <code dir="ltr">days&#95;in&#95;hospital</code> بیشترین همبستگی را با هدف (<code dir="ltr">died</code>) دارد. چرا ممکن است این ستون **نشت داده (leakage)** باشد؟ EDA چگونه به کشف آن کمک می‌کند؟

<details>
<summary><b>پاسخ</b></summary>

الف) روی همین داده، <code dir="ltr">island</code> عملاً برچسب دو گونه را لو می‌دهد؛ اما این رابطه ویژگی **نمونه‌گیری** است (کدام گونه در کدام جزیره بررسی شده)، نه ویژگی بدن پنگوئن. مدلی که به جزیره تکیه کند، برای جزیره‌ی جدید (که در داده‌ی آموزش نبوده) یا اگر گونه‌ای در جزیره‌ی دیگری دیده شود، شکست می‌خورد. اگر هدف تشخیص گونه از روی **اندازه‌ها**ست، <code dir="ltr">island</code> را کنار بگذارید یا جداگانه ارزیابی کنید.

ب) جمله‌ی ۱ را می‌تواند: این یک **توصیف** از همین نمونه است و boxplot و <code dir="ltr">groupby</code> آن را نشان می‌دهند. جمله‌های ۲ و ۳ ادعای **علّی** هستند و EDA نمی‌تواند آن‌ها را ثابت کند: در جمله‌ی ۲، جزیره با گونه گره خورده است (Biscoe خانه‌ی Gentoo سنگین است) و اثرشان قابل تفکیک نیست؛ جمله‌ی ۳ حتی به‌عنوان همبستگی هم گمراه‌کننده است، چون درون هر گونه رابطه **مثبت** است (پارادوکس سیمپسون). EDA **فرضیه** می‌سازد؛ آزمودن آن‌ها کار طراحی آزمایش و استنباط آماری است.

ج) تعداد روزهای بستری فقط **بعد** از رخ دادن نتیجه کامل معلوم می‌شود (بیماری که زود فوت کند، روزهای بستری کمتری دارد) و در لحظه‌ی پیش‌بینی (هنگام پذیرش) در دسترس نیست. مدل روی داده‌ی تاریخی عالی کار می‌کند و در عمل بی‌فایده است. EDA با دو کار کمک می‌کند: (۱) همبستگی «بیش از حد خوب» یک ستون با هدف، زنگ خطر است؛ (۲) برای هر ستون باید پرسید «این مقدار **چه زمانی** ثبت می‌شود؟» که بخشی از مرحله‌ی نمای کلی و شناخت داده است.

</details>

</div>

<div dir="rtl">

## خلاصه‌ی جلسه

- آمار توصیفی یک ستون را با چند عدد خلاصه می‌کند؛ میانگین و انحراف معیار به داده‌ی پرت حساس‌اند و میانه، IQR و MAD مقاوم.
- به تعریف‌ها دقت کنید: <code dir="ltr">ddof</code> در Pandas و NumPy، <code dir="ltr">bias</code> و <code dir="ltr">fisher</code> در SciPy.
- <code dir="ltr">scipy.stats</code> ابزارهای آماده‌ی <code dir="ltr">describe</code>، <code dir="ltr">zscore</code>، <code dir="ltr">median&#95;abs&#95;deviation</code>، <code dir="ltr">pearsonr</code> و <code dir="ltr">spearmanr</code> را دارد؛ قاعده‌ی <code dir="ltr">&#124;z&#124; &gt; 3</code> برای داده‌ی کوچک کار نمی‌کند.
- EDA یک فرایند هشت‌مرحله‌ای است: پرسش، نمای کلی، کیفیت، تک‌متغیره، دومتغیره، چندمتغیره، یافته‌ها و قدم بعد.
- روی داده‌ی پنگوئن‌ها دیدیم که مخلوط شدن گروه‌ها می‌تواند شکل توزیع و حتی علامت همبستگی را عوض کند؛ EDA فرضیه می‌سازد، اما رابطه‌ی علّی را ثابت نمی‌کند.

**جلسه‌ی بعد:** پروژه‌ی پایانی.

</div>